# T-11 — Load Relationships into Neo4j

Load all four relationship types in T-11's scope into the AuraDB instance using the official `neo4j` Python driver. Every load is **idempotent** (uses `MERGE`), so this notebook is safe to re-run.

Requires [`04_load_nodes.ipynb`](04_load_nodes.ipynb) to have run successfully — every relationship below requires both endpoint labels to already exist in the graph.

## Scope

Four relationship types — exactly the ones named in T-11:

| Relationship | Properties | Source | Expected count |
|---|---|---|---|
| `:User-[:RATED]->:Movie` | `rating` (float), `timestamp` (int) | `data/raw/ml-100k/u.data` | 100,000 |
| `:Movie-[:IN_GENRE]->:Genre` | — | `data/raw/ml-100k/u.item` (boolean cols) | ~3,000 |
| `:Movie-[:DIRECTED_BY]->:Director` | — | `data/enriched/enriched_movies.csv` (`directors`) | ~1,700–2,000 |
| `:Movie-[:FEATURES_ACTOR]->:Actor` | `billing_order` (1–10) | `data/enriched/enriched_movies.csv` (`cast`) | ~16,000 |

## Not in scope

- `:Occupation` / `:Gender` nodes and their `HAS_*` edges — deferred to their own task.
- Derived relationships (`COLLABORATED_WITH`, `SIMILAR_TO`) — a later projection step.

## The pattern (same for all four edges)

1. **Python** reads the source file and produces a flat DataFrame of **edge rows** — one row per future relationship.
2. **Cypher** does the rest: `MATCH` both endpoints (constraint-backed → fast), then `MERGE` the edge with its properties.

The Cypher is nearly identical across all four — only the labels and properties change.

## Setup

In [1]:
# Make `from src...` work when running from notebooks/
import sys
from pathlib import Path

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.etl import (
    read_ratings, read_movie_genres, read_movie_directors, read_movie_cast,
    to_records,
)
from src.graph import (
    run_query, run_write_query, run_write_query_chunked, close_driver,
)

## Optional teardown — relationships only

Uncomment **only** if you want to wipe every relationship in the database before reloading edges. Keeps all nodes intact (so you don't have to re-run `04_load_nodes.ipynb`). Destructive — only do this on a throwaway AuraDB instance.

In [2]:
# run_write_query("MATCH ()-[r]->() DELETE r")
# print("All relationships deleted. Nodes preserved.")

## 1 — RATED

100,000 edges from `u.data`. The largest load by an order of magnitude — sent in chunks of 5,000 via `run_write_query_chunked` so we don't stress AuraDB Free's transaction limits and so you can watch progress with `tqdm`.

`rating` is cast to float to match the schema contract; `timestamp` stays as Unix epoch int.

In [3]:
ratings_df = read_ratings()
print(f"Read {len(ratings_df):,} ratings from u.data.")
print(f"  rating range: {ratings_df['rating'].min()}–{ratings_df['rating'].max()}")
ratings_df.head()

Read 100,000 ratings from u.data.
  rating range: 1.0–5.0


,userId,movieId,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
cypher = """
UNWIND $rows AS row
MATCH (u:User  {userId:  row.userId})
MATCH (m:Movie {movieId: row.movieId})
MERGE (u)-[r:RATED]->(m)
SET r.rating = row.rating, r.timestamp = row.timestamp
"""
run_write_query_chunked(cypher, to_records(ratings_df), chunk_size=5000, desc="RATED")

count = run_query("MATCH ()-[:RATED]->() RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count:,} :RATED edges.")

RATED: 100%|██████████| 20/20 [00:24<00:00,  1.22s/chunk]

Neo4j now has 100,000 :RATED edges.


## 2 — IN_GENRE

Built from `u.item`'s 19 boolean genre columns (the canonical ml-100k taxonomy, matching the 19 `:Genre` nodes loaded in T-10). One edge per movie-genre membership; most movies are in 1–3 genres.

In [5]:
movie_genres_df = read_movie_genres()
print(f"Read {len(movie_genres_df):,} movie-genre edge rows from u.item.")
print(f"  avg genres per movie: {len(movie_genres_df) / movie_genres_df['movieId'].nunique():.2f}")
movie_genres_df.head()

Read 2,893 movie-genre edge rows from u.item.
  avg genres per movie: 1.72


,movieId,genre
0,267,unknown
1,1373,unknown
2,2,Action
3,4,Action
4,17,Action


In [6]:
cypher = """
UNWIND $rows AS row
MATCH (m:Movie {movieId: row.movieId})
MATCH (g:Genre {name:    row.genre})
MERGE (m)-[:IN_GENRE]->(g)
"""
run_write_query(cypher, {"rows": to_records(movie_genres_df)})

count = run_query("MATCH ()-[:IN_GENRE]->() RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count:,} :IN_GENRE edges.")

Neo4j now has 2,893 :IN_GENRE edges.


## 3 — DIRECTED_BY

Built by exploding the `directors` column from `enriched_movies.csv`. Most movies have one director, but anthology films and co-directed films (the Coens, the Wachowskis) produce multiple edges per movie. Unmatched movies are silently skipped.

In [7]:
directors_df = read_movie_directors()
print(f"Read {len(directors_df):,} movie-director edge rows from enriched_movies.csv.")
print(f"  unique movies with at least one director: {directors_df['movieId'].nunique():,}")
directors_df.head()

Read 1,790 movie-director edge rows from enriched_movies.csv.
  unique movies with at least one director: 1,678


,movieId,name
0,1,John Lasseter
1,2,Martin Campbell
2,3,Allison Anders
3,3,Alexandre Rockwell
4,3,Quentin Tarantino


In [8]:
cypher = """
UNWIND $rows AS row
MATCH (m:Movie    {movieId: row.movieId})
MATCH (d:Director {name:    row.name})
MERGE (m)-[:DIRECTED_BY]->(d)
"""
run_write_query(cypher, {"rows": to_records(directors_df)})

count = run_query("MATCH ()-[:DIRECTED_BY]->() RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count:,} :DIRECTED_BY edges.")

Neo4j now has 1,790 :DIRECTED_BY edges.


## 4 — FEATURES_ACTOR

Built by exploding the `cast` column from `enriched_movies.csv`. Each matched movie contributes up to 10 edges, one per top-billed actor. `billing_order` (1 = top billing) comes from the list index, since TMDB already sorts cast by billing.

In [9]:
cast_df = read_movie_cast()
print(f"Read {len(cast_df):,} movie-actor edge rows from enriched_movies.csv.")
print(f"  avg cast size per matched movie: {len(cast_df) / cast_df['movieId'].nunique():.2f}")
cast_df.head()

Read 16,243 movie-actor edge rows from enriched_movies.csv.
  avg cast size per matched movie: 9.70


,movieId,name,billing_order
0,1,Tom Hanks,1
1,1,Tim Allen,2
2,1,Don Rickles,3
3,1,Jim Varney,4
4,1,Wallace Shawn,5


In [10]:
cypher = """
UNWIND $rows AS row
MATCH (m:Movie {movieId: row.movieId})
MATCH (a:Actor {name:    row.name})
MERGE (m)-[r:FEATURES_ACTOR]->(a)
SET r.billing_order = row.billing_order
"""
run_write_query(cypher, {"rows": to_records(cast_df)})

count = run_query("MATCH ()-[:FEATURES_ACTOR]->() RETURN count(*) AS n")[0]["n"]
print(f"Neo4j now has {count:,} :FEATURES_ACTOR edges.")

Neo4j now has 16,243 :FEATURES_ACTOR edges.


## Summary — edge counts by relationship type

Final tally across every relationship type in the database.

In [11]:
result = run_query("""
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(*) AS n
ORDER BY n DESC
""")
print(f"{'relationship':<20} {'count':>10}")
print("-" * 32)
for row in result:
    print(f"{row['relationship']:<20} {row['n']:>10,}")

relationship              count
--------------------------------
RATED                   100,000
FEATURES_ACTOR           16,243
IN_GENRE                  2,893
DIRECTED_BY               1,790


## Spot-checks — what the graph can now do

Two queries to confirm the graph is actually traversable. These are the kinds of questions a tabular schema would need three or more JOINs to answer.

### Query 1 — User 196's high-rating genre affinity

Two hops in three lines of Cypher: `User → RATED (4★+) → Movie → IN_GENRE → Genre`. The graph collapses what would be a multi-join SQL query into a single traversal.

In [12]:
result = run_query("""
MATCH (u:User {userId: 196})-[r:RATED]->(m:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= 4
RETURN g.name AS genre, count(*) AS high_ratings
ORDER BY high_ratings DESC LIMIT 5
""")
print(f"{'genre':<20} {'high_ratings':>12}")
print("-" * 34)
for row in result:
    print(f"{row['genre']:<20} {row['high_ratings']:>12}")

genre                high_ratings
----------------------------------
Comedy                         18
Drama                           7
Romance                         6
War                             1
Children's                      1


### Query 2 — Full neighborhood of *Toy Story* (movieId=1)

Pulls the director, top 3 actors (by billing), and all genres in one traversal. This is what the graph buys you: a movie's full context with no joins, no aggregations, just `MATCH`.

In [13]:
result = run_query("""
MATCH (m:Movie {movieId: 1})
OPTIONAL MATCH (m)-[:DIRECTED_BY]->(d:Director)
OPTIONAL MATCH (m)-[fa:FEATURES_ACTOR]->(a:Actor)
OPTIONAL MATCH (m)-[:IN_GENRE]->(g:Genre)
RETURN m.title                          AS title,
       collect(DISTINCT d.name)         AS directors,
       collect(DISTINCT a.name)[..3]    AS top_actors,
       collect(DISTINCT g.name)         AS genres
""")
row = result[0]
print(f"Title:     {row['title']}")
print(f"Directors: {row['directors']}")
print(f"Actors:    {row['top_actors']}")
print(f"Genres:    {row['genres']}")

Title:     Toy Story
Directors: ['John Lasseter']
Actors:    ['Annie Potts', 'Don Rickles', 'Erik von Detten']
Genres:    ['Animation', "Children's", 'Comedy']


## Idempotency check (re-run after the cells above)

Run the summary cell again. Counts should be **identical** — no double-loads. That's the `MERGE` pattern doing its job, same as in T-10.

In [14]:
close_driver()
print("Driver closed.")

Driver closed.
